# Edge-AI Inference Demo — Kria KV260

ARM (PYNQ) host driving CV32E40P RISC-V soft-core + Conv-CNN accelerator. Notebook structured in 4 sections:

| Section | Run frequency | Purpose |
|---------|---------------|---------|
| **1. Setup**       | Once per kernel session    | Paths → overlay → meta → firmware → weights → buffers |
| **2. Per-image**   | Re-run for each new image  | Preprocess → kick → decode → visualize |
| **3. Batch helper**| Optional                   | `predict(path)` convenience wrapper |
| **4. Cleanup**     | Once at end                | Free CMA buffers |

**Memory map** (xem [CLAUDE.md](../../CLAUDE.md#memory-map)): I-BRAM Port B `0xA000_0000` · D-BRAM Port B `0xB004_0000` · conv_cnn ctrl `0xB001_0000` · DDR coherent qua S_AXI_HPC0_FPD.

**Shared register layout** (offset từ D-BRAM Port B base):

| Offset | Reg | Direction | Mô tả |
|--------|-----|-----------|-------|
| `+0x00` | `CMD_FROM_ARM`   | ARM → RISC-V | `0x01`=START, `0x00`=idle |
| `+0x04` | `STATUS_TO_ARM`  | RISC-V → ARM | `0x00`=IDLE / `0x01`=BUSY / `0x02`=DONE |
| `+0x08` | `DATASET_ID`     | ARM → RISC-V | INRIA=0, cats_dogs=1 |
| `+0x0C` | `RESULT_CLASS`   | RISC-V → ARM | (RISC-V option, ARM tự argmax thay thế) |
| `+0x10` | `RESULT_CONF`    | RISC-V → ARM | (RISC-V option) |
| `+0x18` | `IFM_PHYS_ADDR`  | ARM → RISC-V | DDR phys addr buffer A |
| `+0x1C` | `OFM_PHYS_ADDR`  | ARM → RISC-V | DDR phys addr buffer B |
| `+0x20` | `WEIGHT_BASE`    | ARM → RISC-V | DDR phys addr weights blob |


---
# 1. SETUP &nbsp;&nbsp;<sub>(run once per kernel session)</sub>


## 1.1 Configure paths

Edit the constants below if your `deploy.sh` target differs from `/home/ubuntu/edgeai/`. The cell after this asserts each file exists — **fail fast** if anything's missing.


In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from pynq import allocate

# Make `from edge_ai import ...` work regardless of cwd
sys.path.insert(0, str(Path.cwd().parent))

PROJECT_ROOT = Path("/home/ubuntu/edgeai")
BITSTREAM    = PROJECT_ROOT / "hw/artifacts/kria_soc_wrapper.bit"
FIRMWARE     = PROJECT_ROOT / "firmware/firmware.bin"
WEIGHTS      = PROJECT_ROOT / "training/export/vgg-tiny_cats_dogs.weights.bin"
IMAGE        = PROJECT_ROOT / "samples/dog_01.jpg"


In [ ]:
# Validate everything exists before touching hardware
missing = [p for p in (BITSTREAM, FIRMWARE, WEIGHTS, IMAGE) if not p.exists()]
for p in (BITSTREAM, FIRMWARE, WEIGHTS, IMAGE):
    print(f"  {'OK ' if p.exists() else 'MISS'}  {p}")
assert not missing, f"Missing files: {missing}\nRun host/scripts/deploy.sh first."


## 1.2 Load bitstream → `EdgeAIOverlay`

`pynq.Overlay()` loads the `.bit` (and the matching `.hwh` handoff metadata, must be co-located with same name) and exposes each Vivado IP as a Python attribute. `EdgeAIOverlay` wraps this and exposes only the IPs we care about (`i_bram`, `d_bram`, `_gpio`).


In [ ]:
from edge_ai import EdgeAIOverlay
overlay = EdgeAIOverlay(str(BITSTREAM))


In [ ]:
# Sanity check: confirm the IPs we expect are present
expected = {"axi_bram_ctrl_2", "axi_bram_ctrl_3", "axi_gpio_0",
            "axi_dma_0", "conv_cnn_0", "riscv_top_0"}
present = set(overlay.overlay.ip_dict.keys())
missing_ips = expected - present
print("IPs present :", sorted(present & expected))
print("IPs missing :", sorted(missing_ips) or "(none)")
assert not missing_ips, f"Bitstream missing IPs: {missing_ips}. Re-synth Vivado project."


## 1.3 Load model metadata

`meta.json` is generated by `training/train.py` and pairs with `weights.bin`. It carries:
- input quantization (scale, zero_point) used during preprocessing
- final OFM shape + scale + zp + which ping-pong buffer it lands in
- max tensor bytes (for ping-pong sizing)
- label list + dataset_id


In [ ]:
from edge_ai import load_meta
meta = load_meta(str(WEIGHTS))
fpga = meta["fpga"]


In [ ]:
print(f"Model      : {meta['model']}")
print(f"Dataset    : {meta['dataset']}  (id={meta['dataset_id']})")
print(f"Labels     : {meta['labels']}")
print(f"Input quant: scale={meta['input']['scale']:.6f}  zp={meta['input']['zero_point']}")
print(f"Layers     : {fpga['num_layers']}")
print(f"Final OFM  : shape={fpga['final_ofm_shape']}  buf_idx={fpga['final_ofm_buf_idx']}")
print(f"            scale={fpga['final_ofm_scale']:.4f}  zp={fpga['final_ofm_zp']}")
print(f"Max tensor : {fpga['max_tensor_bytes']} B  (use as ping-pong buffer size)")


## 1.4 Boot RISC-V soft-core

Three ops, in this order:
1. **Clear shared regs**: zero out 8 handshake registers in D-BRAM Port B → no stale state from previous run.
2. **Stream `firmware.bin` into I-BRAM Port B**: 4-byte word writes starting at offset 0. RISC-V boot vector is `0x0000_0000` (I-BRAM Port A from RISC-V side).
3. **Pulse reset GPIO**: `axi_gpio_0[0]` 0 (halt) → 1 (run). CV32E40P begins fetching at `0x0`.

After this, `STATUS_TO_ARM` should read 0 (firmware sets it to BUSY only when ARM kicks).


In [ ]:
overlay.clear_shared_regs()
fw_bytes = overlay.load_firmware(str(FIRMWARE))   # also pulses reset GPIO
print(f"Loaded {fw_bytes} B firmware  →  I-BRAM @ 0xA000_0000")
print(f"Status reg after boot: 0x{overlay.read_status():08X}  (expect 0x00 = IDLE)")


## 1.5 Stage weights to DDR

`pynq.allocate()` returns a CMA-backed `numpy` array with `.physical_address` — DMA can DMA-read this directly without `mmap`/`ioctl` boilerplate. The S_AXI_HPC0_FPD path through the CCI-400 keeps it cache-coherent with ARM.

`flush()` after the bulk copy ensures any dirty cache lines hit DDR before we hand the address to RISC-V.


In [ ]:
from edge_ai import load_weights_blob
wbytes = load_weights_blob(str(WEIGHTS))
wbuf = allocate(shape=(len(wbytes),), dtype=np.uint8)
wbuf[:] = np.frombuffer(wbytes, dtype=np.uint8)
wbuf.flush()


In [ ]:
overlay.set_weights_addr(wbuf.physical_address)
print(f"Weights blob: {len(wbytes)} B")
print(f"  CMA phys  : 0x{wbuf.physical_address:08X}")
print(f"  REG_WEIGHT_BASE programmed (RISC-V adds layer.weight_offset for per-layer addr)")


## 1.6 Allocate ping-pong DDR buffers

`buf_a` is layer 0 input (we'll fill it per-image), `buf_b` is the scratch counterpart. RISC-V toggles them: layer `i` reads from `(i&1) ? buf_b : buf_a` and writes to the other. Final OFM ends up in `buf_a` (idx=0) if `NUM_LAYERS` is odd, else `buf_b` (idx=1) — `meta['fpga']['final_ofm_buf_idx']` tells us which.

Buffer size = `max_tensor_bytes` from meta — enough to hold the largest IFM/OFM in the network.


In [ ]:
pp_size = fpga["max_tensor_bytes"]
buf_a = allocate(shape=(pp_size,), dtype=np.int8)
buf_b = allocate(shape=(pp_size,), dtype=np.int8)


In [ ]:
overlay.set_io_buffers(buf_a.physical_address, buf_b.physical_address)
overlay.set_dataset_id(meta["dataset_id"])
print(f"buf_a  phys=0x{buf_a.physical_address:08X}  ({pp_size} B)  ← layer 0 input")
print(f"buf_b  phys=0x{buf_b.physical_address:08X}  ({pp_size} B)  ← scratch")
print(f"DATASET_ID = {meta['dataset_id']} ({meta['dataset']})")


## 1.7 Setup verification

Single diagnostic cell — read back every shared register to confirm the host-side programming actually landed. If anything here is wrong, **stop and debug** before kicking RISC-V.


In [ ]:
from edge_ai import constants as C
def dump_regs():
    for name, off in [("CMD_FROM_ARM",   C.REG_CMD_FROM_ARM),
                      ("STATUS_TO_ARM",  C.REG_STATUS_TO_ARM),
                      ("DATASET_ID",     C.REG_DATASET_ID),
                      ("IFM_PHYS_ADDR",  C.REG_IFM_PHYS_ADDR),
                      ("OFM_PHYS_ADDR",  C.REG_OFM_PHYS_ADDR),
                      ("WEIGHT_BASE",    C.REG_WEIGHT_BASE)]:
        print(f"  D-BRAM[+{off:#04x}] {name:<14} = 0x{overlay.d_bram.read(off):08X}")
dump_regs()
assert overlay.read_status() == C.STATUS_IDLE, "RISC-V not idle — re-run 1.4"
assert overlay.d_bram.read(C.REG_WEIGHT_BASE) == wbuf.physical_address
assert overlay.d_bram.read(C.REG_IFM_PHYS_ADDR) == buf_a.physical_address
assert overlay.d_bram.read(C.REG_OFM_PHYS_ADDR) == buf_b.physical_address
print("\n[OK] Setup verified.")


---
# 2. PER-IMAGE INFERENCE &nbsp;&nbsp;<sub>(re-run cells 2.1–2.5 for each new image)</sub>

To test another image: change `IMAGE` in cell 1.1, then re-run **only** the cells in this section. Setup state stays valid as long as the kernel is alive.


## 2.1 Load + preprocess

cv2 reads BGR uint8 → convert RGB → resize to `meta['input']['fpga_size']` (128×128 for vgg-tiny) → normalize [0,1] → INT8 quantize via `q = round(x/scale) + zp`.


In [ ]:
from edge_ai import preprocess_image
ifm_int8 = preprocess_image(str(IMAGE), meta)
print(f"IFM shape={ifm_int8.shape}  dtype={ifm_int8.dtype}  "
      f"min={ifm_int8.min()}  max={ifm_int8.max()}")


## 2.2 Visualize input

Quick sanity check — left = original, right = de-quantized 128×128 (what FPGA actually sees). Big visual difference here ≠ bug; it's down-sampling artifacts.


In [ ]:
import cv2
orig = cv2.cvtColor(cv2.imread(str(IMAGE)), cv2.COLOR_BGR2RGB)
disp = ((ifm_int8.astype(np.int32) - meta["input"]["zero_point"])
        * meta["input"]["scale"]).clip(0, 1)

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(orig);  axes[0].set_title(f"Original {orig.shape[:2]}");          axes[0].axis("off")
axes[1].imshow(disp);  axes[1].set_title(f"INT8 quantized {ifm_int8.shape[:2]}"); axes[1].axis("off")
plt.tight_layout(); plt.show()


## 2.3 DMA IFM → kick RISC-V → poll done

Three things in one cell because they happen back-to-back and we want one wall-clock measurement:
1. Copy preprocessed IFM into `buf_a`, `flush()` to DDR.
2. Write `CMD_START` to `REG_CMD_FROM_ARM`.
3. Spin-poll `REG_STATUS_TO_ARM` until `STATUS_DONE` (timeout = 5 s).


In [ ]:
flat = ifm_int8.reshape(-1)
buf_a[:flat.size] = flat
buf_a.flush()

t0 = time.perf_counter()
overlay.kick()
overlay.poll_done(timeout_s=5.0)
latency_ms = (time.perf_counter() - t0) * 1000

overlay.reset_cmd()                  # so next kick is fresh
print(f"DONE in {latency_ms:.2f} ms  ({1000/latency_ms:.1f} fps if back-to-back)")


## 2.4 Decode final OFM → label

Pick the correct ping-pong buffer (per `final_ofm_buf_idx`), `invalidate()` so the CPU sees the freshly DMA-written data instead of cache, reshape to `[H, W, C]`, then `gap_argmax()` does:
1. dequantize: `real = (q - zp) * scale`
2. global average pool over (H, W)
3. softmax for human-readable confidence
4. `argmax` for class id


In [ ]:
from edge_ai import gap_argmax
final_buf = buf_a if fpga["final_ofm_buf_idx"] == 0 else buf_b
final_buf.invalidate()

H, W, C = fpga["final_ofm_shape"][-3:]
ofm = np.frombuffer(final_buf[:H*W*C], dtype=np.int8).reshape(H, W, C)
cls, conf_pct, logits = gap_argmax(ofm, fpga["final_ofm_scale"], fpga["final_ofm_zp"])
label = meta["labels"][cls] if cls < len(meta["labels"]) else f"<unknown:{cls}>"

print(f"Logits     : {logits}")
print(f"Prediction : {label}  (class={cls})")
print(f"Confidence : {conf_pct:.1f}%")
print(f"Latency    : {latency_ms:.2f} ms")


## 2.5 Visualize result


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(orig)
ax.set_title(f"{label}  ({conf_pct:.1f}% — {latency_ms:.1f} ms)", fontsize=14)
ax.axis("off"); plt.show()


---
# 3. BATCH HELPER &nbsp;&nbsp;<sub>(optional)</sub>

If you want to test many images quickly without re-running the whole Section 2 each time, define a single function that captures the per-image flow. Reuses `overlay`, `meta`, `buf_a`, `buf_b` from Section 1.


In [ ]:
def predict(img_path):
    """Run inference on one image. Returns (label, conf_pct, latency_ms)."""
    ifm = preprocess_image(str(img_path), meta)
    flat = ifm.reshape(-1)
    buf_a[:flat.size] = flat; buf_a.flush()

    t0 = time.perf_counter()
    overlay.kick(); overlay.poll_done(timeout_s=5.0)
    lat = (time.perf_counter() - t0) * 1000
    overlay.reset_cmd()

    fb = buf_a if fpga["final_ofm_buf_idx"] == 0 else buf_b
    fb.invalidate()
    H, W, C = fpga["final_ofm_shape"][-3:]
    ofm = np.frombuffer(fb[:H*W*C], dtype=np.int8).reshape(H, W, C)
    cls, conf, _ = gap_argmax(ofm, fpga["final_ofm_scale"], fpga["final_ofm_zp"])
    return meta["labels"][cls], conf, lat


In [ ]:
# Example: run on a few sample images. Adjust glob pattern as needed.
for img in sorted((PROJECT_ROOT / "samples").glob("*.jpg"))[:5]:
    label, conf, lat = predict(img)
    print(f"  {img.name:30}  →  {label:8}  ({conf:5.1f}%, {lat:5.1f} ms)")


---
# 4. CLEANUP

Reset `CMD_FROM_ARM` so RISC-V doesn't see stale request, then return CMA buffers. Skipping this is *usually* fine (kernel cleans up), but the CMA pool is finite (~256 MB by default) so leaks compound across sessions.


In [ ]:
overlay.reset_cmd()
for b in (wbuf, buf_a, buf_b):
    b.freebuffer()
print("Cleaned up CMA buffers; CMD_FROM_ARM = 0.")
